In [1]:
# Step 1: Load the original data and H matrix
import numpy as np
import pandas as pd
import pickle
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc, precision_score, recall_score, f1_score, matthews_corrcoef
import matplotlib.pyplot as plt

# Load the original dataset and H matrix
original_df_path = "./data/df_4th.csv"
validation_df_path = "./data/df_3rd.csv"
df_ori = pd.read_csv(original_df_path, index_col=0)
df_val_ori = pd.read_csv(validation_df_path, index_col=0, encoding='cp1252')

In [3]:
# Add this code after loading the validation dataset in step4

# Create baseline characteristics table for validation cohort (2015 data)
def create_validation_baseline_table(df_val_ori):
    """
    Create baseline characteristics table for validation cohort
    matching the format of Table 1 from the main study
    """
    
    # Split by MM or better status
    mm_or_better = df_val_ori[df_val_ori['MMorbetter'] == 1]
    non_mm = df_val_ori[df_val_ori['MMorbetter'] == 0]
    total = df_val_ori
    
    def calculate_stats(data, column, is_categorical=False):
        if is_categorical:
            # For categorical data, return counts and percentages
            counts = data[column].value_counts()
            total_count = len(data)
            result = {}
            for value, count in counts.items():
                percentage = (count / total_count) * 100
                result[value] = f"{count} ({percentage:.1f}%)"
            return result
        else:
            # For continuous data, return median (IQR)
            median_val = data[column].median()
            q1 = data[column].quantile(0.25)
            q3 = data[column].quantile(0.75)
            return f"{median_val:.0f} ({q1:.0f}, {q3:.0f})"
    
    # Create results dictionary
    results = {}
    
    # Total N
    results['Total N'] = {
        'Total': len(total),
        'non MM': len(non_mm), 
        'MM or better': len(mm_or_better)
    }
    
    # Sex (assuming 1=male, 0=female based on typical coding)
    sex_total = calculate_stats(total, 'Sex', True)
    sex_non_mm = calculate_stats(non_mm, 'Sex', True)
    sex_mm = calculate_stats(mm_or_better, 'Sex', True)
    
    results['Sex (Male)'] = {
        'Total': sex_total.get(1, '0 (0.0%)'),
        'non MM': sex_non_mm.get(1, '0 (0.0%)'),
        'MM or better': sex_mm.get(1, '0 (0.0%)')
    }
    
    # Age
    results['Age'] = {
        'Total': calculate_stats(total, 'Age'),
        'non MM': calculate_stats(non_mm, 'Age'),
        'MM or better': calculate_stats(mm_or_better, 'Age')
    }
    
    # Onset age
    results['Onset age'] = {
        'Total': calculate_stats(total, 'Onset_age'),
        'non MM': calculate_stats(non_mm, 'Onset_age'),
        'MM or better': calculate_stats(mm_or_better, 'Onset_age')
    }
    
    # MG subtypes (assuming: 1=OMG, 2=EOMG, 3=LOMG, 4=TAMG, 5=MuSK, 6=SNMG)
    subtype_mapping = {1: 'OMG', 2: 'EOMG', 3: 'LOMG', 4: 'TAMG', 5: 'MuSK', 6: 'SNMG'}
    
    for subtype_code, subtype_name in subtype_mapping.items():
        subtype_total = len(total[total['MG_subtypes'] == subtype_code])
        subtype_non_mm = len(non_mm[non_mm['MG_subtypes'] == subtype_code])
        subtype_mm = len(mm_or_better[mm_or_better['MG_subtypes'] == subtype_code])
        
        total_pct = (subtype_total / len(total)) * 100
        non_mm_pct = (subtype_non_mm / len(non_mm)) * 100 if len(non_mm) > 0 else 0
        mm_pct = (subtype_mm / len(mm_or_better)) * 100 if len(mm_or_better) > 0 else 0
        
        results[subtype_name] = {
            'Total': f"{subtype_total} ({total_pct:.1f}%)",
            'non MM': f"{subtype_non_mm} ({non_mm_pct:.1f}%)",
            'MM or better': f"{subtype_mm} ({mm_pct:.1f}%)"
        }
    
    # Clinical scores
    for score_name, column_name in [('MGC total score', 'MGC'), 
                                    ('MGADL total score', 'MGADL'),
                                    ('MGQOL15r total score', 'MGQOL15')]:
        if column_name in total.columns:
            results[score_name] = {
                'Total': calculate_stats(total, column_name),
                'non MM': calculate_stats(non_mm, column_name),
                'MM or better': calculate_stats(mm_or_better, column_name)
            }
    
    # PSL dose
    if 'PSL_dose' in total.columns:
        results['PSL dose'] = {
            'Total': calculate_stats(total, 'PSL_dose'),
            'non MM': calculate_stats(non_mm, 'PSL_dose'),
            'MM or better': calculate_stats(mm_or_better, 'PSL_dose')
        }
    
    return results

# Calculate MG scores before creating the table
# MGADL total score
mgadl_columns = ["MGADL_speech", "MGADL_chewing", "MGADL_swallowing", "MGADL_respiration",
                "MGADL_toothbrushing", "MGADL_getting_up", "MGADL_diplopia", "MGADL_ptosis"]
if all(col in df_val_ori.columns for col in mgadl_columns):
    df_val_ori["MGADL"] = df_val_ori[mgadl_columns].sum(axis=1)

# MGC total score (same as QMG in this study)
mgc_columns = ["MGC_ptosis", "MGC_diplopia", "MGC_eyelid_closure", "MGC_speech", "MGC_chewing",
               "MGC_swallowing", "MGC_respiration", "MGC_neck", "MGC_upper_limb", "MGC_lower_limb"]
if all(col in df_val_ori.columns for col in mgc_columns):
    df_val_ori["MGC"] = df_val_ori[mgc_columns].sum(axis=1)

# MGQOL15r total score
mgqol_columns = ["MGQOL1_dissatisfaction", "MGQOL2_seeing", "MGQOL3_eating", "MGQOL4_social_activity_restriction",
                "MGQOL5_hobby_entertainment", "MGQOL6_family_role", "MGQOL7_behavior_modification",
                "MGQOL8_work_impact", "MGQOL9_speaking", "MGQOL10_driving", "MGQOL11_feeling_down",
                "MGQOL12_walking", "MGQOL13_quick_action", "MGQOL14_mental_crushing", "MGQOL15_dressing"]
if all(col in df_val_ori.columns for col in mgqol_columns):
    df_val_ori["MGQOL15"] = df_val_ori[mgqol_columns].sum(axis=1)

# Apply the function to create the table
validation_baseline_stats = create_validation_baseline_table(df_val_ori)

# Print the results in a formatted way
print("Supplementary Table S1. Baseline characteristics of the 2015 validation cohort")
print("=" * 80)
print(f"{'Characteristic':<25} {'Total (N={})'.format(validation_baseline_stats['Total N']['Total']):<20} {'non MM (N={})'.format(validation_baseline_stats['Total N']['non MM']):<20} {'MM or better (N={})'.format(validation_baseline_stats['Total N']['MM or better']):<20}")
print("-" * 80)

for characteristic, values in validation_baseline_stats.items():
    if characteristic != 'Total N':
        print(f"{characteristic:<25} {values['Total']:<20} {values['non MM']:<20} {values['MM or better']:<20}")

# Create a DataFrame for easier export
validation_table_df = pd.DataFrame(validation_baseline_stats).T
validation_table_df.to_csv('./out/supplementary_table_s1.csv')
print(f"\nTable saved to './out/supplementary_table_s1.csv'")

# Statistical comparisons (optional)
from scipy.stats import mannwhitneyu, chi2_contingency

print("\n" + "="*50)
print("Statistical comparisons between non MM and MM or better groups:")
print("="*50)

# Continuous variables
for var in ['Age', 'Onset_age', 'MGC', 'MGADL', 'MGQOL15', 'PSL_dose']:
    if var in df_val_ori.columns:
        non_mm_vals = df_val_ori[df_val_ori['MMorbetter'] == 0][var].dropna()
        mm_vals = df_val_ori[df_val_ori['MMorbetter'] == 1][var].dropna()
        
        if len(non_mm_vals) > 0 and len(mm_vals) > 0:
            stat, p_value = mannwhitneyu(non_mm_vals, mm_vals)
            print(f"{var}: p = {p_value:.3f}")

# Categorical variables
# Sex
if 'Sex' in df_val_ori.columns:
    contingency_sex = pd.crosstab(df_val_ori['Sex'], df_val_ori['MMorbetter'])
    chi2, p_sex, dof, expected = chi2_contingency(contingency_sex)
    print(f"Sex: p = {p_sex:.3f}")

# MG subtypes
if 'MG_subtypes' in df_val_ori.columns:
    contingency_subtypes = pd.crosstab(df_val_ori['MG_subtypes'], df_val_ori['MMorbetter'])
    chi2, p_subtypes, dof, expected = chi2_contingency(contingency_subtypes)
    print(f"MG subtypes: p = {p_subtypes:.3f}")

Supplementary Table S1. Baseline characteristics of the 2015 validation cohort
Characteristic            Total (N=414)        non MM (N=161)       MM or better (N=253)
--------------------------------------------------------------------------------
Sex (Male)                147 (35.5%)          42 (26.1%)           105 (41.5%)         
Age                       65 (51, 75)          65 (53, 76)          64 (50, 75)         
Onset age                 47 (33, 60)          47 (34, 60)          47 (31, 59)         
OMG                       74 (17.9%)           19 (11.8%)           55 (21.7%)          
EOMG                      111 (26.8%)          45 (28.0%)           66 (26.1%)          
LOMG                      86 (20.8%)           32 (19.9%)           54 (21.3%)          
TAMG                      90 (21.7%)           36 (22.4%)           54 (21.3%)          
MuSK                      8 (1.9%)             3 (1.9%)             5 (2.0%)            
SNMG                      45 (10.9%)   